In [5]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [6]:
df=pd.read_csv("train.csv")

In [7]:
df.shape

(595212, 59)

In [8]:
df.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595212 entries, 0 to 595211
Data columns (total 59 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              595212 non-null  int64  
 1   target          595212 non-null  int64  
 2   ps_ind_01       595212 non-null  int64  
 3   ps_ind_02_cat   595212 non-null  int64  
 4   ps_ind_03       595212 non-null  int64  
 5   ps_ind_04_cat   595212 non-null  int64  
 6   ps_ind_05_cat   595212 non-null  int64  
 7   ps_ind_06_bin   595212 non-null  int64  
 8   ps_ind_07_bin   595212 non-null  int64  
 9   ps_ind_08_bin   595212 non-null  int64  
 10  ps_ind_09_bin   595212 non-null  int64  
 11  ps_ind_10_bin   595212 non-null  int64  
 12  ps_ind_11_bin   595212 non-null  int64  
 13  ps_ind_12_bin   595212 non-null  int64  
 14  ps_ind_13_bin   595212 non-null  int64  
 15  ps_ind_14       595212 non-null  int64  
 16  ps_ind_15       595212 non-null  int64  
 17  ps_ind_16_

In [10]:
df.isnull().sum().sum()

0

In [11]:
df.duplicated().sum()

0

In [12]:
df['target'].value_counts()

target
0    573518
1     21694
Name: count, dtype: int64

In [13]:
df['target'].value_counts(normalize=True)*100

target
0    96.355248
1     3.644752
Name: proportion, dtype: float64

In [25]:
# The id column is an identifier, not a meaningful predictive feature.
# so exclude it
x=df.drop(['target','id'],axis=1)
y=df['target']

print("x shape:",x.shape)

x shape: (595212, 57)


### Separate x and y

In [26]:
### Train-Test Split
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("x_train:",x_train.shape)
print("x_test:",x_test.shape)
print("y_train:",y_train.shape)
print("y_test:",y_test.shape)


x_train: (476169, 57)
x_test: (119043, 57)
y_train: (476169,)
y_test: (119043,)


### Preprocessing

In [27]:
cat_cols = [col for col in x.columns if '_cat' in col]
bin_cols = [col for col in x.columns if '_bin' in col]
num_cols = [col for col in x.columns if col not in cat_cols + bin_cols]

print("Categorical columns:", len(cat_cols))
print("Binary columns:", len(bin_cols))
print("Numerical columns:", len(num_cols))

Categorical columns: 14
Binary columns: 17
Numerical columns: 26


In [29]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

In [31]:
# cat → One-Hot Encodes the 14 categorical columns
# num → Standardizes the 26 numerical columns
# remainder='passthrough' → keeps the 17 binary columns unchanged.
# So, not scaling the binary 0/1 variables,

In [35]:
# fitting the preprocessing only on training data
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)
# fit_transform() → only X_train
#transform() → X_test

In [37]:
print("x_train_processed shape:", x_train_processed.shape)
print("x_test_processed shape:", x_test_processed.shape)

x_train_processed shape: (476169, 227)
x_test_processed shape: (119043, 227)


In [38]:
# The increase from 57 → 227 is expected because the 14 categorical columns were expanded into multiple one-hot encoded columns.

In [45]:
# Checking that there are no missing values in processed data
print("Missing values in x_train_processed:", np.isnan(x_train_processed.data).sum())
print("Missing values in x_test_processed:", np.isnan(x_test_processed.data).sum())

Missing values in x_train_processed: 0
Missing values in x_test_processed: 0
